# Homework 2: Advanced Efficient Inference

In this homework, you will implement and analyze several techniques used to make LLM inference more efficient.
We will repeat multiple notions from the lecture: Quantization, Pruning, MoEs, KV Cache, sparsity, and more!

In [1]:
# DO NOT MODIFY THIS CELL

import math
import random
import numpy as np
import matplotlib.pyplot as plt

from typing import List, Tuple, Dict, Optional

np.set_printoptions(precision=4, suppress=True)

SEED = 42
rng = np.random.default_rng(SEED)
random.seed(SEED)


## Part 1: Symmetric vs. Asymmetric Quantization

In the lecture, you have learned about symmetric and asymmetric quantization. In this task, you will implement both types of quantization on given matrices and compare their errors.

In [2]:
# DO NOT MODIFY THIS CELL

# HELPER FUNCTIONS

def mean_abs_error(A: np.ndarray, B: np.ndarray) -> float:
    return float(np.mean(np.abs(A - B)))

def max_abs_error(A: np.ndarray, B: np.ndarray) -> float:
    return float(np.max(np.abs(A - B)))

### 1.1) Implement symmetric int8 quantization

First, you will implement symmetric quantization. We want to clip quantized values to the int8 range `[-128,127]`.

Please use the following formulas:
$$
\text{scale} = \frac{\max(|W|)}{127}
$$

$$
W_q = \operatorname{round}(\frac{W}{\text{scale}})
$$

$$
\hat{W} = W_q \cdot \text{scale}
$$

and complete the functions below:

In [3]:
# TODO: IMPLEMENT THIS CELL

def quantize_symmetric_int8(W: np.ndarray) -> Tuple[np.ndarray, float]:
    """
    Symmetrically quantize W to int8.

    Args:
        W: floating-point NumPy array

    Returns:
        W_q: int8 quantized array
        scale: positive floating-point scale
    """
    scale = np.max(np.abs(W))
    W_q = np.round(W/scale)
    return np.int8(W_q),scale


def dequantize_symmetric_int8(W_q: np.ndarray, scale: float) -> np.ndarray:
    return W_q * scale 

### 1.2) Implement asymmetric uint8 quantization

We will implement asymmetric quantization using unsigned integers (uint). The quantized values should be clipped to the uint8 range `[0,255]`.

Use the following formulas:

$$
\text{scale} = \frac{\max(W) - \min(W)}{255}
$$

$$
\text{zero\_point} = \operatorname{round}\left(-\frac{\min(W)}{\text{scale}}\right)
$$

$$
W_q = \operatorname{round}\left(\frac{W}{\text{scale}} + \text{zero\_point}\right)
$$

$$
\hat{W} = (W_q - \text{zero\_point}) \cdot \text{scale}
$$



Clip `W_q` to `[0, 255]` after rounding, and clip `zero_point` to `[0, 255]`. The provided **positive** matrices are nonnegative with `min(W)=0`, so the zero point maps the smallest value to bin 0.

Implement:


In [4]:
# TODO: IMPLEMENT THIS CELL

def quantize_asymmetric_uint8(W: np.ndarray) -> Tuple[np.ndarray, float, int]:
    """
    Asymmetrically quantize W to uint8.
    """
    scale = (np.max(W) - np.min(W))/255 
    zero_point = np.round(-np.min(W)/scale)
    W_q = np.round(W / scale + zero_point)
    return np.uint8(W_q),scale,zero_point

def dequantize_asymmetric_uint8(W_q: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    """
    Dequantize back.
    """
    W = (W_q - zero_point) * scale
    return W


In [5]:
# DO NOT MODIFY THIS CELL

W_test = np.array([
    [0.00, 0.20, 0.40, 0.80],
    [0.00, 0.25, 0.60, 1.10],
    [0.00, 0.30, 0.70, 1.20],
], dtype=np.float32)

W_sym_q, sym_scale = quantize_symmetric_int8(W_test)
W_sym_hat = dequantize_symmetric_int8(W_sym_q, sym_scale)

W_asym_q, asym_scale, zero_point = quantize_asymmetric_uint8(W_test)
W_asym_hat = dequantize_asymmetric_uint8(W_asym_q, asym_scale, zero_point)
assert W_sym_q.dtype == np.int8
assert W_asym_q.dtype == np.uint8
assert W_sym_hat.shape == W_test.shape
assert W_asym_hat.shape == W_test.shape
assert sym_scale > 0
assert asym_scale > 0
assert 0 <= zero_point <= 255

print("Symmetric scale:", sym_scale)
print("Asymmetric scale:", asym_scale)
print("Zero point:", zero_point)

print("Symmetric MAE:", mean_abs_error(W_test, W_sym_hat))
print("Asymmetric MAE:", mean_abs_error(W_test, W_asym_hat))


Symmetric scale: 1.2
Asymmetric scale: 0.0047058826
Zero point: -0.0
Symmetric MAE: 0.2291666865348816
Asymmetric MAE: 0.0007353077526204288


### 1.3) Compare quantization schemes.

Now, apply both quantization schemes on the following three matrices. Then, compare the reconstruction errors by computing Mean Absolute Error (MAE) and Maximum Error below.

In [6]:
# DO NOT MODIFY THIS CELL

W_zero_centered = np.array([
    [-1.0, -0.5,  0.0,  0.5],
    [ 1.0, -0.8,  0.3, -0.2],
    [ 0.7, -0.6,  0.2, -0.1],
], dtype=np.float32)

W_positive = np.array([
    [0.00, 0.20, 0.40, 0.80],
    [0.00, 0.25, 0.60, 1.10],
    [0.00, 0.30, 0.70, 1.20],
], dtype=np.float32)

W_outlier = np.array([
    [0.00, 0.20, 0.40, 20.0],
    [0.00, 0.25, 0.60, 1.10],
    [0.00, 0.30, 0.70, 1.20],
], dtype=np.float32)


In [7]:
# TODO: IMPLEMENT THIS CELL

def compare_symmetric_asymmetric(W: np.ndarray) -> Dict[str, float]:
    """
    Quantize W symmetrically and asymmetrically and return reconstruction errors.

    Returns dictionary with:
        sym_mae
        sym_max_error
        asym_mae
        asym_max_error
    """
    W_q_s,scale = quantize_symmetric_int8(W)
    W_q_a,scale,zero_point = quantize_asymmetric_uint8(W)
    W_r_s = dequantize_symmetric_int8(W_q_s,scale)
    W_r_a = dequantize_asymmetric_uint8(W_q_a,scale,zero_point)
    sym_mae = mean_abs_error(W,W_r_s)
    asym_mae = mean_abs_error(W,W_r_a)
    sym_max_error = np.max(np.abs(W-W_r_s))
    asym_max_error = np.max(np.abs(W-W_r_a))
    d = {
        "sym_mae" : sym_mae,
        "sym_max_error" : sym_max_error,
        "asym_mae" : asym_mae,
        "asym_max_error" : asym_max_error,
    }
    return d 


for name, W in [
    ("zero_centered", W_zero_centered),
    ("positive", W_positive),
    ("outlier", W_outlier),
]:
    result = compare_symmetric_asymmetric(W)
    print(name, result)

zero_centered {'sym_mae': 0.4883986711502075, 'sym_max_error': np.float32(0.99215686), 'asym_mae': 0.0024509758222848177, 'asym_max_error': np.float32(0.0039215684)}
positive {'sym_mae': 0.4609313905239105, 'sym_max_error': np.float32(1.1952941), 'asym_mae': 0.0007353077526204288, 'asym_max_error': np.float32(0.002352953)}
outlier {'sym_mae': 2.055964231491089, 'sym_max_error': np.float32(19.921568), 'asym_mae': 0.010866012424230576, 'asym_max_error': np.float32(0.03529413)}


ADD BRIEF ANALYSIS HERE

Both the mean squared error and max error are larger in the symmetric quantization compared to asymmetric.
The strongest error can be seen in the outlier case, where the symmetric quantization has a much higher error compared to asymmetric quantization.


### 1.4) Short Questions

* **Q1a** Why can asymmetric quantization work better for tensors whose values are mostly positive?
* **Q1b** What is the role of the zero point?
* **Q1c** Why can a single outlier increase the reconstruction error for many other values?

### Your Answers Here

Use this cell to write down your answers (1-2 sentences each) to the questions.

Q1a: Because it offers a higher precision, in the case of uint8 it maps the maximum to 255 and the minimum to 0.
Symmetric quantization maps the absolute highest value of the original range to min and max, which means that 
one side of the range is wasted if the distribution is scewed. 

Q1b: 
Since our range needs to be [0,255] we use zeropoint = round(min(W)/s) as an offset.  
To convert it back to the int8 aysmmetric representation [min(W)/s,max(W)/s] , which is directly scalable to the original range, we store the zero point inbetween quantization and dequantization.  

Q1c: Because, a single outlier will lead to a high range of values, even though most points are much closer to 0. 
This leads to most values being represented very closely in the quantized space, which results in a loss of precision. 


## Part 2: SmoothQuant Channel Scaling

The lecture introduced SmoothQuant as a method for moving quantization difficulty from activations to weights. 
SmoothQuant rescales each input channel:

$$
\bar{X}_{:,j} = \frac{X_{:,j}}{s_j}
$$

$$
\bar{W}_{j,:} = W_{j,:} \cdot s_j
$$

This preserves the matrix product before quantization:

$$
XW = \bar{X}\bar{W}
$$

The channel-wise scale is:

$$
s_j = \frac{\max(|X_j|)^\alpha}{\max(|W_j|)^{1-\alpha}}
$$

where $\alpha \in [0, 1]$.

Here $|X_j|$ is the max absolute value in column $j$ of $X$. For $W$ with shape `[D_in, D_out]`, $|W_j|$ is the max absolute value in **row** $j$ (`max(abs(W), axis=1)`).


### 2.1) Implement SmoothQuant scaling

Implement the functions `compute_smoothquant_scales` and `apply_smoothquant`.

In [8]:
# DO NOT MODIFY THIS CELL

X_calib = np.array([
    [ 0.5,  20.0,  0.2, -0.1],
    [ 0.4, -18.0,  0.3,  0.0],
    [ 0.6,  22.0, -0.2,  0.1],
    [ 0.5, -21.0,  0.1, -0.2],
], dtype=np.float32)

X_test = np.array([
    [ 0.4,  19.0,  0.3, -0.2],
    [ 0.6, -20.0, -0.1,  0.1],
], dtype=np.float32)

W_smooth_base = np.array([
    [ 0.5, -0.3,  0.2],
    [ 0.1,  0.2, -0.1],
    [ 0.4, -0.5,  0.3],
    [-0.2,  0.1,  0.6],
], dtype=np.float32)

In [9]:
# TODO: IMPLEMENT THIS CELL

def compute_smoothquant_scales(
    X_calib: np.ndarray,
    W: np.ndarray,
    alpha: float,
    eps: float = 1e-8,
) -> np.ndarray:
    """
    Compute SmoothQuant-style channel scales.

    Args:
        X_calib: calibration activations, shape [num_samples, D_in]
        W: weight matrix, shape [D_in, D_out]
        alpha: smoothing parameter in [0, 1]
        eps: small value to avoid division by zero

    Returns:
        scales: shape [D_in]
    """
    scales = (np.max(np.abs(X_calib), axis=0)**alpha) / (np.max(np.abs(W),axis=1) ** (1-alpha))
    return scales

def apply_smoothquant(
    X: np.ndarray,
    W: np.ndarray,
    scales: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Apply SmoothQuant-style scaling.

    Returns:
        X_smooth: X divided channel-wise by scales
        W_smooth: W multiplied row-wise by scales
    """
    scales_diag = np.multiply(scales,np.eye(N=scales.shape[0]))
    S_inv = np.linalg.inv(scales_diag)
    X_smooth = S_inv @ X.T
    X_smooth = X_smooth.T
    W_smooth = scales_diag @ W 
    return (X_smooth, W_smooth)

In [10]:
# DO NOT MODIFY THIS CELL

alpha = 0.5
scales = compute_smoothquant_scales(X_calib, W_smooth_base, alpha)
X_test_smooth, W_smooth = apply_smoothquant(X_test, W_smooth_base, scales)
print(X_test)
print(X_test_smooth)
print(W_test)
print(W_smooth)

Y_original = X_test @ W_smooth_base
Y_smooth = X_test_smooth @ W_smooth

assert scales.shape == (W_smooth_base.shape[0],)
assert np.all(scales > 0)
assert X_test_smooth.shape == X_test.shape
assert W_smooth.shape == W_smooth_base.shape
assert np.allclose(Y_original, Y_smooth, atol=1e-5)


[[  0.4  19.    0.3  -0.2]
 [  0.6 -20.   -0.1   0.1]]
[[ 0.3651  1.8116  0.3873 -0.3464]
 [ 0.5477 -1.9069 -0.1291  0.1732]]
[[0.   0.2  0.4  0.8 ]
 [0.   0.25 0.6  1.1 ]
 [0.   0.3  0.7  1.2 ]]
[[ 0.5477 -0.3286  0.2191]
 [ 1.0488  2.0976 -1.0488]
 [ 0.3098 -0.3873  0.2324]
 [-0.1155  0.0577  0.3464]]


### 2.2) Quantize before and after smoothing

Now quantize both activations and weights symmetrically. Compare the output error for different `alpha` values.

In [11]:
# TODO: IMPLEMENT THIS CELL

def quantized_matmul_symmetric(X: np.ndarray, W: np.ndarray) -> np.ndarray:
    """
    Quantize X and W symmetrically to int8, dequantize them, then compute X_hat @ W_hat.
    """
    X_q,scale_x = quantize_symmetric_int8(X)
    W_q,scale_w = quantize_symmetric_int8(W)
    X_hat = dequantize_symmetric_int8(X_q,scale_x)
    W_hat = dequantize_symmetric_int8(W_q,scale_w)
    return X_hat @ W_hat 

def evaluate_smoothquant_alphas(
    X_calib: np.ndarray,
    X_test: np.ndarray,
    W: np.ndarray,
    alphas: List[float],
) -> Dict[float, float]:
    """
    For each alpha:
    - compute SmoothQuant scales using X_calib and W
    - smooth X_test and W
    - quantize both smoothed matrices
    - compute MAE against full precision X_test @ W

    Returns:
        dictionary mapping alpha to MAE
    """
    res = {}
    for alpha in alphas: 
        scales = compute_smoothquant_scales(X_calib, W,alpha)
        X_smooth, W_smooth = apply_smoothquant(X_test,W,scales)
        print("alpha: ", alpha)
        print(X_smooth)
        print(W_smooth)
        XW = quantized_matmul_symmetric(X_test,W)
        XW_smooth = quantized_matmul_symmetric(X_smooth,W_smooth)
        mae = mean_abs_error(XW,XW_smooth)
        res[alpha] = mae
    return res

In [12]:
# DO NOT MODIFY THIS CELL

baseline_quantized = quantized_matmul_symmetric(X_test, W_smooth_base)
baseline_error = mean_abs_error(Y_original, baseline_quantized)

alpha_errors = evaluate_smoothquant_alphas(
    X_calib=X_calib,
    X_test=X_test,
    W=W_smooth_base,
    alphas=[0.0, 0.25, 0.5, 0.75, 1.0],
)

print("Baseline quantization error without smoothing:", baseline_error)
for alpha, err in alpha_errors.items():
    print(f"alpha={alpha}: MAE={err}")

assert all(np.isfinite(v) for v in alpha_errors.values())

alpha:  0.0
[[ 0.2   3.8   0.15 -0.12]
 [ 0.3  -4.   -0.05  0.06]]
[[ 1.     -0.6     0.4   ]
 [ 0.5     1.     -0.5   ]
 [ 0.8    -1.      0.6   ]
 [-0.3333  0.1667  1.    ]]
alpha:  0.25
[[ 0.2702  2.6237  0.241  -0.2039]
 [ 0.4054 -2.7618 -0.0803  0.1019]]
[[ 0.7401 -0.444   0.296 ]
 [ 0.7242  1.4483 -0.7242]
 [ 0.4979 -0.6223  0.3734]
 [-0.1962  0.0981  0.5886]]
alpha:  0.5
[[ 0.3651  1.8116  0.3873 -0.3464]
 [ 0.5477 -1.9069 -0.1291  0.1732]]
[[ 0.5477 -0.3286  0.2191]
 [ 1.0488  2.0976 -1.0488]
 [ 0.3098 -0.3873  0.2324]
 [-0.1155  0.0577  0.3464]]
alpha:  0.75
[[ 0.4934  1.2508  0.6223 -0.5886]
 [ 0.7401 -1.3167 -0.2074  0.2943]]
[[ 0.4054 -0.2432  0.1621]
 [ 1.519   3.038  -1.519 ]
 [ 0.1928 -0.241   0.1446]
 [-0.068   0.034   0.2039]]
alpha:  1.0
[[ 0.6667  0.8636  1.     -1.    ]
 [ 1.     -0.9091 -0.3333  0.5   ]]
[[ 0.3  -0.18  0.12]
 [ 2.2   4.4  -2.2 ]
 [ 0.12 -0.15  0.09]
 [-0.04  0.02  0.12]]
Baseline quantization error without smoothing: 2.6083333492279053
alpha=0.0: M

### Discussion

Analyze and discuss your results in two sentences. What is the impact of the parameter `alpha`, what does it control? You can use the next markdown cell for your answer.

ADD BRIEF ANALYSIS HERE.
Alpha = 0 completely smooths the weights, and shifts all of the difficulty to the activations and the higher the alpha the more difficulty gets shifted to the weights. 

## Part 3: Quantizing a Real Hugging Face Model

In this part, you will load a tiny causal language model and compare outputs before and after quantization.

You may choose between two options on how to do the quantization (based on whether you have a GPU available).

1. [bitsandbytes 8-bit quantization](https://huggingface.co/docs/transformers/en/quantization/bitsandbytes), if your environment supports it. E.g., if you have a GPU or are using Google Colab. 
2. PyTorch dynamic quantization, which should work on CPU for many environments.

### 3.1) Load full-precision model

First, we will load a full-precision model from Huggingface in order to compare our quantized model to it. For that, implement the `generate_greedy` function.

In [13]:
# TODO: RUN THIS CELL

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_fp = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model_fp.eval()

prompts = [
    "The capital of France is",
    "In a transformer, attention is",
]

c:\Users\Ramneek\anaconda3\envs\llm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 8055.98it/s]


In [14]:
# TODO: IMPLEMENT THIS CELL

def generate_greedy(model, tokenizer, prompt: str, max_new_tokens: int = 20) -> str:
    """
    Generate text greedily from a causal LM.
    """
    output = prompt
    for _ in range(max_new_tokens):
        tokenized_input = tokenizer(output, return_tensors="pt")
        input_ids = tokenized_input["input_ids"].to(device)  
        with torch.no_grad():
            logits = model(input_ids).logits
        logits_next_token = logits[:,-1,:]
        prob_dist = torch.softmax(logits_next_token, dim = 1)
        next_token_id = torch.argmax(prob_dist)
        next_token = tokenizer.convert_ids_to_tokens(next_token_id.unsqueeze(-1))
        next_token = tokenizer.convert_tokens_to_string(next_token)
        output += next_token
    return output




In [15]:
# DO NOT MODIFY THIS CELL

for prompt in prompts:
    out = generate_greedy(model_fp, tokenizer, prompt)
    print("PROMPT:", prompt)
    print("OUTPUT:", out)
    print()

PROMPT: The capital of France is
OUTPUT: The capital of France is Paris. It is the largest city in Europe and the third largest in the world. It is also

PROMPT: In a transformer, attention is
OUTPUT: In a transformer, attention is paid to the insulation resistance of the windings and the insulation resistance of the core. The insulation resistance



### 3.2) OPTION A: bitsandbytes quantization

If your environment supports it, use bitsandbytes to download the same model in 8bit.

In [16]:
# TODO: YOU MAY NEED TO SPECIFY THE GPU PATH

# If this cell fails, use OPTION B instead.

model_quant = None

try:
    from transformers import BitsAndBytesConfig
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
    model_quant = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-0.5B",
        device_map = "auto",
        quantization_config = quant_config
    )

except Exception as e:
    model_quant = None
    print("Error:", repr(e))


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 352.59it/s]


### OPTION B: PyTorch dynamic quantization fallback

Use this if bitsandbytes does not work.
Use Torch Quantization via `quantize_dynamic` to quantize the model manually. You can find the documentation [here](https://docs.pytorch.org/docs/2.12/quantization-support.html). 

In [17]:
if model_quant is None:
    import copy

    # Deep-copy so you do not modify model_fp in place
    model_quant = copy.deepcopy(model_fp)
    model_quant = torch.quantization.quantize_dynamic(
        model_quant,
        {torch.nn.Linear},
        dtype=torch.qint8,
    )
    model_quant.eval()


In [18]:
# DO NOT MODIFY THIS CELL

for prompt in prompts:
    out_fp = generate_greedy(model_fp, tokenizer, prompt)
    out_quant = generate_greedy(model_quant, tokenizer, prompt)

    print("PROMPT:", prompt)
    print("FULL PRECISION:", out_fp)
    print("QUANTIZED:", out_quant)
    print("IDENTICAL?", out_fp == out_quant)
    print()

c:\Users\Ramneek\anaconda3\envs\llm_env\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


PROMPT: The capital of France is
FULL PRECISION: The capital of France is Paris. It is the largest city in Europe and the third largest in the world. It is also
QUANTIZED: The capital of France is Paris. It is the largest city in Europe and the third largest in the world. It is also
IDENTICAL? True

PROMPT: In a transformer, attention is
FULL PRECISION: In a transformer, attention is paid to the insulation resistance of the windings and the insulation resistance of the core. The insulation resistance
QUANTIZED: In a transformer, attention is paid to the fact that the core is not allowed to be grounded. The purpose of this is to
IDENTICAL? False



### Discussion

Analyze and discuss your results in two sentences. Which quantization path did you use? Did it change the generated text? Why might it or might it not? You can use the next markdown cell for your answer.

YOUR BRIEF ANSWER HERE.

The quantization path from Option A was used and it changed the output of the second prompt. It changed the prompt because the matrices have reconstruction errors, when going from quantized to dequantized. 


## Part 4: Pruning

In this task, you will implement magnitude pruning and 2:4 structured pruning.

In [19]:
# DO NOT MODIFY THIS CELL

rng = np.random.default_rng(SEED)

W_prune = rng.normal(loc=0.0, scale=1.0, size=(8, 8)).astype(np.float32)
X_prune = rng.normal(loc=0.0, scale=1.0, size=(16, 8)).astype(np.float32)

Y_prune_ref = X_prune @ W_prune

print("W_prune:")
print(W_prune)

W_prune:
[[ 0.3047 -1.04    0.7505  0.9406 -1.951  -1.3022  0.1278 -0.3162]
 [-0.0168 -0.853   0.8794  0.7778  0.066   1.1272  0.4675 -0.8593]
 [ 0.3688 -0.9589  0.8785 -0.0499 -0.1849 -0.6809  1.2225 -0.1545]
 [-0.4283 -0.3521  0.5323  0.3654  0.4127  0.4308  2.1416 -0.4064]
 [-0.5122 -0.8138  0.616   1.129  -0.1139 -0.8402 -0.8245  0.6506]
 [ 0.7433  0.5432 -0.6655  0.2322  0.1167  0.2187  0.8714  0.2236]
 [ 0.6789  0.0676  0.2891  0.6313 -1.4572 -0.3197 -0.4704 -0.6389]
 [-0.2751  1.4949 -0.8658  0.9683 -1.6829 -0.3349  0.1628  0.5862]]


### 4.1) Implement Magnitude Pruning and Sparsity Metric

Fill in the following functions.

In [ ]:
# TODO: IMPLEMENT THIS CELL

def compute_sparsity(W: np.ndarray) -> float:
    """
    Return the fraction of entries equal to zero.
    """
    # YOUR CODE HERE
    zero_count = W.size - np.count_nonzero(W)
    return   zero_count / W.size 


def magnitude_prune(W: np.ndarray, sparsity: float) -> np.ndarray:
    """
    Unstructured magnitude pruning.

    Args:
        W: weight matrix
        sparsity: target fraction of weights to prune

    Returns:
        W_pruned with approximately sparsity fraction of entries set to zero.
    """
    # YOUR CODE HERE
    W_copy = W.reshape((-1),copy=True)
    W_copy = np.sort(np.abs(W_copy))
    idx = np.round(sparsity * len(W_copy)).astype(np.int16)
    threshold = W_copy[idx]
    pruned_W = np.multiply(W, (np.abs(W) >= threshold))
    return pruned_W


[[ 0.3047 -1.04    0.7505  0.9406 -1.951  -1.3022  0.1278 -0.3162]
 [-0.0168 -0.853   0.8794  0.7778  0.066   1.1272  0.4675 -0.8593]
 [ 0.3688 -0.9589  0.8785 -0.0499 -0.1849 -0.6809  1.2225 -0.1545]
 [-0.4283 -0.3521  0.5323  0.3654  0.4127  0.4308  2.1416 -0.4064]
 [-0.5122 -0.8138  0.616   1.129  -0.1139 -0.8402 -0.8245  0.6506]
 [ 0.7433  0.5432 -0.6655  0.2322  0.1167  0.2187  0.8714  0.2236]
 [ 0.6789  0.0676  0.2891  0.6313 -1.4572 -0.3197 -0.4704 -0.6389]
 [-0.2751  1.4949 -0.8658  0.9683 -1.6829 -0.3349  0.1628  0.5862]]
Magnitude sparsity: 0.0
Magnitude sparsity: 0.5


### 4.2) Implement 2:4 structured pruning

For each row, process weights in consecutive groups of 4. In every group of 4, keep the 2 weights with largest absolute value and set the other 2 weights to zero.

You may assume the number of columns is divisible by 4.

In [116]:
# TODO: IMPLEMENT THIS CELL

def prune_row(row_element: np.ndarray):
    row_element = row_element.reshape(-1,4)
    for cell_element in row_element:
        two_lowest = np.argsort(np.abs(cell_element))[:2]
        cell_element[two_lowest] = 0
    row_element = row_element.reshape(-1)
    return row_element

def structured_2_to_4_prune(W: np.ndarray) -> np.ndarray:
    """
    Apply 2:4 structured pruning row-wise.

    For each row and each consecutive group of 4 columns:
    keep the 2 entries with largest absolute value and set the other 2 to zero.
    """
    W_pruned = np.copy(W)
    W_pruned = np.apply_along_axis(prune_row,axis=1,arr=W_pruned)
    return W_pruned
print(W_prune)
W_24 = structured_2_to_4_prune(W_prune)
print(W_24)
print(W_24.reshape(-1,4))


[[ 0.3047 -1.04    0.7505  0.9406 -1.951  -1.3022  0.1278 -0.3162]
 [-0.0168 -0.853   0.8794  0.7778  0.066   1.1272  0.4675 -0.8593]
 [ 0.3688 -0.9589  0.8785 -0.0499 -0.1849 -0.6809  1.2225 -0.1545]
 [-0.4283 -0.3521  0.5323  0.3654  0.4127  0.4308  2.1416 -0.4064]
 [-0.5122 -0.8138  0.616   1.129  -0.1139 -0.8402 -0.8245  0.6506]
 [ 0.7433  0.5432 -0.6655  0.2322  0.1167  0.2187  0.8714  0.2236]
 [ 0.6789  0.0676  0.2891  0.6313 -1.4572 -0.3197 -0.4704 -0.6389]
 [-0.2751  1.4949 -0.8658  0.9683 -1.6829 -0.3349  0.1628  0.5862]]
[[ 0.     -1.04    0.      0.9406 -1.951  -1.3022  0.      0.    ]
 [ 0.     -0.853   0.8794  0.      0.      1.1272  0.     -0.8593]
 [ 0.     -0.9589  0.8785  0.      0.     -0.6809  1.2225  0.    ]
 [-0.4283  0.      0.5323  0.      0.      0.4308  2.1416  0.    ]
 [ 0.     -0.8138  0.      1.129   0.     -0.8402 -0.8245  0.    ]
 [ 0.7433  0.     -0.6655  0.      0.      0.      0.8714  0.2236]
 [ 0.6789  0.      0.      0.6313 -1.4572  0.      0.     -0.

In [118]:
# DO NOT MODIFY THIS CELL

W_mag = magnitude_prune(W_prune, sparsity=0.5)
W_24 = structured_2_to_4_prune(W_prune)

assert W_mag.shape == W_prune.shape
assert W_24.shape == W_prune.shape

assert abs(compute_sparsity(W_mag) - 0.5) <= 0.05
assert abs(compute_sparsity(W_24) - 0.5) <= 1e-6

# Check 2:4 pattern.
for row in range(W_24.shape[0]):
    for start in range(0, W_24.shape[1], 4):
        group = W_24[row, start:start+4]
        assert np.count_nonzero(group) == 2

print("Magnitude sparsity:", compute_sparsity(W_mag))
print("2:4 sparsity:", compute_sparsity(W_24))


Magnitude sparsity: 0.5
2:4 sparsity: 0.5


### 4.3) Compare output errors of pruning methods

Next, you will compute the L2 error between both methods.

In [123]:
# TODO: IMPLEMENT THIS CELL

def relative_l2_error(Y_ref: np.ndarray, Y_approx: np.ndarray, eps: float = 1e-8) -> float:
    """
    Return ||Y_ref - Y_approx||_2 / ||Y_ref||_2.
    """
    return  np.linalg.norm(x=(Y_ref - Y_approx), ord=2) / np.linalg.norm(x=(Y_ref), ord=2)


Y_mag = X_prune @ W_mag
Y_24 = X_prune @ W_24

mag_mae = mean_abs_error(Y_prune_ref, Y_mag)
prune24_mae = mean_abs_error(Y_prune_ref, Y_24)

mag_rel = relative_l2_error(Y_prune_ref, Y_mag)
prune24_rel = relative_l2_error(Y_prune_ref, Y_24)

print("Magnitude pruning MAE:", mag_mae)
print("2:4 pruning MAE:", prune24_mae)
print("Magnitude pruning relative L2:", mag_rel)
print("2:4 pruning relative L2:", prune24_rel)

Magnitude pruning MAE: 0.49762386083602905
2:4 pruning MAE: 0.5465976595878601
Magnitude pruning relative L2: 0.33517146
2:4 pruning relative L2: 0.31237212


### 4.4) Short Questions

* **Q4a** Why might pruning small-magnitude weights preserve output quality?
* **Q4b** Why is unstructured sparsity not automatically faster on GPUs?
* **Q4c** Why is 2:4 sparsity more hardware-friendly?

### Your Answers Here

Use this cell to write down your answers (1-2 sentences each) to the questions.

Q4a: The attention mechanism inherently focuses largely on high attention weighted input tokens and downweighs inputs with lower scores. Low attention scores are a direct result of having lower weights in W_k,W_q and W_v 
Q4b: The GPU can more effectively parallelize and store matrices in structured pruning approaches. 
Q4c: Because the data can be compressed where the non zero values and their positions can be stored as 2 bits indicies. The GPU can save the space for the zero values.  

## Part 5: Toy Mixture-of-Experts Router

In the lecture example, router logits such as `[1, 2, 0.5, 3]` are used to select the top-2 experts, then the selected expert outputs are combined using softmax weights.

In [143]:
# DO NOT MODIFY THIS CELL

rng = np.random.default_rng(SEED)

num_tokens = 6
d_model = 4
d_out = 3
num_experts = 4
top_k = 2

X_moe = rng.normal(size=(num_tokens, d_model)).astype(np.float32)
router_W = rng.normal(size=(d_model, num_experts)).astype(np.float32)

experts = [
    rng.normal(size=(d_model, d_out)).astype(np.float32)
    for _ in range(num_experts)
]

print("X_moe shape:", X_moe.shape)
print("router_W shape:", router_W.shape)
print("number of experts:", len(experts))
print(router_W)

X_moe shape: (6, 4)
router_W shape: (4, 4)
number of experts: 4
[[-0.4283 -0.3521  0.5323  0.3654]
 [ 0.4127  0.4308  2.1416 -0.4064]
 [-0.5122 -0.8138  0.616   1.129 ]
 [-0.1139 -0.8402 -0.8245  0.6506]]


### 5.1) Implement routing helpers

First, you will implement helper functions to identify the top k indices, apply softmax, and apply the expert to the token representation.

In [138]:
# TODO: IMPLEMENT THIS CELL

def top_k_indices(logits: np.ndarray, k: int) -> np.ndarray:
    """
    Return indices of the top-k logits in descending order.
    """
    return np.argsort(logits)[::-1][:k]


def softmax_1d(x: np.ndarray) -> np.ndarray:
    """
    Numerically stable softmax for a 1D array.
    """
    max = np.max(x)
    return np.exp(x - max) / np.sum(np.exp(x - max))

def expert_forward(x: np.ndarray, expert_matrix: np.ndarray) -> np.ndarray:
    """
    Apply one linear expert to one token representation.
    """
    # YOUR CODE HERE
    return x @ expert_matrix

example_logits = np.array([1.0, 2.0, 0.5, 3.0])
selected = top_k_indices(example_logits, 2)
print(selected)
assert selected.tolist() == [3, 1]

weights = softmax_1d(example_logits[selected])
print(weights)

[3 1]
[0.7311 0.2689]


In [139]:
# DO NOT MODIFY THIS CELL

example_logits = np.array([1.0, 2.0, 0.5, 3.0])
selected = top_k_indices(example_logits, 2)
print(selected)
assert selected.tolist() == [3, 1]

weights = softmax_1d(example_logits[selected])
print(weights)
assert np.allclose(weights.sum(), 1.0)
assert weights[0] > weights[1]

x_example = np.array([1.0, 2.0])
W_example = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
])
assert np.allclose(expert_forward(x_example, W_example), x_example)

[3 1]
[0.7311 0.2689]


### 5.2) Implement MoE forward pass

Next, we compute a toy MoE layer.

For each token x:
- compute router logits x @ router_W
- select top-k experts
- softmax over the selected logits
- compute each selected expert output
- return the weighted sum of selected expert outputs

Also, complete the `compute_expert_load` function which counts how often each expert is selected.

In [164]:
# TODO: IMPLEMENT THIS CELL

def moe_forward(
    X: np.ndarray,
    router_W: np.ndarray,
    experts: List[np.ndarray],
    k: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute a toy MoE layer.

    Returns:
        outputs: shape [num_tokens, d_out]
        assignments: shape [num_tokens, k], selected expert indices
        router_weights: shape [num_tokens, k], softmax weights over selected experts
    """
    # YOUR CODE HERE
    num_tokens = X.shape[0]
    d_out = experts[0].shape[1]
    router_logits = X @ router_W #shape num_tokens, num_experts
    assignments = np.zeros(shape=(num_tokens,k))
    for i,token_row in enumerate(router_logits): 
        assignments[i,:] = top_k_indices(token_row, k=k)
    print(assignments)
    router_weights = np.zeros(shape=assignments.shape)
    for i,token_top_k in enumerate(assignments): 
        print(token_top_k)
        router_weights[i,:] = softmax_1d(router_logits[i,token_top_k.astype(np.int16)])

    outputs = np.zeros(shape=(num_tokens,d_out))
    token_output = np.zeros(shape=(1,d_out))
    for i,token_top_k in enumerate(assignments): 
        token_output = np.zeros(shape=(1,d_out))
        for j, expert_idx in enumerate(token_top_k):
            token_output += router_weights[i,j] * expert_forward(X[i,:],expert_matrix=experts[expert_idx.astype(np.int16)])
        outputs[i,:] = token_output
    return outputs,assignments,router_weights

def compute_expert_load(assignments: np.ndarray, num_experts: int) -> np.ndarray:
    """
    Count how often each expert is selected.
    """
    # YOUR CODE HERE
    count_array = np.zeros(shape=num_experts)
    for i in range(num_experts):
        count_array[i] = np.sum(assignments == i)
    return count_array



In [165]:
# DO NOT MODIFY THIS CELL

moe_outputs, assignments, router_weights = moe_forward(
    X_moe, router_W, experts, k=top_k
)

expert_load = compute_expert_load(assignments, num_experts)

assert moe_outputs.shape == (num_tokens, d_out)
assert assignments.shape == (num_tokens, top_k)
assert router_weights.shape == (num_tokens, top_k)
assert np.allclose(router_weights.sum(axis=1), np.ones(num_tokens))
assert expert_load.sum() == num_tokens * top_k

print("Assignments:")
print(assignments)
print("Router weights:")
print(router_weights)
print("Expert load:", expert_load)

[[3. 0.]
 [1. 0.]
 [3. 0.]
 [2. 1.]
 [3. 0.]
 [3. 2.]]
[3. 0.]
[1. 0.]
[3. 0.]
[2. 1.]
[3. 0.]
[3. 2.]
Assignments:
[[3. 0.]
 [1. 0.]
 [3. 0.]
 [2. 1.]
 [3. 0.]
 [3. 2.]]
Router weights:
[[0.9545 0.0455]
 [0.5047 0.4953]
 [0.9384 0.0616]
 [0.9335 0.0665]
 [0.9229 0.0771]
 [0.8971 0.1029]]
Expert load: [4. 2. 2. 4.]


### 5.3) Short Questions

* **Q5a** Why does MoE increase total model capacity without activating all parameters for each token?
* **Q5b** Why does MoE not necessarily reduce GPU memory?
* **Q5c** What is top-k routing?
* **Q5d** What problem can load imbalance cause?

### Your Answers Here

Use this cell to write down your answers (1-2 sentences each) to the questions.
Q5a: 
Q5b: Because the multiple experts have to loaded simultaniously into the memory, to allow switching at inference between tokens. 
Q5c: Instead of selecting one expert per token, we select the k best experts selected by the router and sum their outputs weighted by a softmax. 
Q5d:  

## Part 6: Sparse Attention Masks

In this part, you will implement and compare several attention masks.

We use the following convention:
* `mask[i, j] = 1 if token i may attend to token j`
* `mask[i, j] = 0 otherwise`

### 6.1) Implement masks

Implement the following masks:
1. Causal Mask
2. Sliding Window
3. Dilated Sliding Window Causal Mask

Also implement `add_global_tokens(mask, global_tokens)` (every query may attend to listed global key positions).

Count how many attention entries are allowed in each mask.

In [ ]:
# TODO: IMPLEMENT THIS CELL

def full_causal_mask(L: int) -> np.ndarray:
    """
    Full causal attention mask.
    token i may attend to token j iff j <= i.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def sliding_window_causal_mask(L: int, window: int) -> np.ndarray:
    """
    Sliding-window causal mask.
    token i may attend to at most the previous `window` tokens, including itself.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def dilated_sliding_window_causal_mask(L: int, window: int, dilation: int) -> np.ndarray:
    """
    Dilated sliding-window causal mask.

    token i attends to:
        i, i-dilation, i-2*dilation, ...
    up to `window` total attended positions.
    """
    # YOUR CODE HERE
    raise NotImplementedError

def add_global_tokens(mask: np.ndarray, global_tokens: List[int]) -> np.ndarray:
    """
    Return a copy of mask where every query row can attend to each global key index.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def count_allowed(mask: np.ndarray) -> int:
    """
    Count allowed attention entries.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY THIS CELL

mask_full = full_causal_mask(4)
expected_full = np.array([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 1, 0],
    [1, 1, 1, 1],
])

assert np.array_equal(mask_full, expected_full)
assert count_allowed(mask_full) == 10

mask_slide = sliding_window_causal_mask(4, window=2)
expected_slide = np.array([
    [1, 0, 0, 0],
    [1, 1, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 1, 1],
])
assert np.array_equal(mask_slide, expected_slide)
assert count_allowed(mask_slide) == 7

mask_dilated = dilated_sliding_window_causal_mask(6, window=3, dilation=2)
assert mask_dilated[5, 5] == 1
assert mask_dilated[5, 3] == 1
assert mask_dilated[5, 1] == 1
assert mask_dilated[5, 4] == 0


### 6.2) Plot mask heatmaps

Now, you are going to plot heatmaps of different attention masks for `L=32`.

Also implement `add_global_tokens(mask, global_tokens)` so every query row can attend to the listed global key positions (use it for the sliding-window and dilated masks below).


In [ ]:
# TODO: IMPLEMENT THIS CELL

L = 32
window = 4
dilation = 2
global_tokens = [0]

masks = {
    "full causal": full_causal_mask(L),
    "sliding window": sliding_window_causal_mask(L, window),
    "dilated sliding": dilated_sliding_window_causal_mask(L, window, dilation),
}

# Plot one heatmap per mask.
# YOUR CODE HERE

### 6.3) Plot Complexity Scaling

For sequence lengths `[16, 32, 64, 128, 256]`, plot the number of allowed attention entries for each mask.

In [ ]:
# TODO: IMPLEMENT THIS CELL

lengths = [16, 32, 64, 128, 256]
window = 16
dilation = 4
global_tokens = [0]

# Compute and plot allowed attention entries vs sequence length.
# YOUR CODE HERE

## Part 7: Attention Sparsity and KV-Cache Policies

In this part, you will load a tiny model, inspect its attention matrices, and implement cache policies over token indices.

This part uses a real model only to observe attention patterns. Your cache policies will be implemented from scratch over token positions, not by modifying the model internals.

### 7.1) Load model with attention outputs

In [ ]:
# TODO: RUN THIS CELL

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

attn_model_name = "Qwen/Qwen2.5-0.5B"

attn_tokenizer = AutoTokenizer.from_pretrained(attn_model_name)
attn_model = AutoModelForCausalLM.from_pretrained(attn_model_name)
attn_model.eval()

prompt = (
    "Large language models use attention to decide which previous tokens "
    "are important for predicting the next token."
)

inputs = attn_tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = attn_model(**inputs, output_attentions=True)

print("Number of layers with attention:", len(outputs.attentions))
print("Shape of first attention tensor:", outputs.attentions[0].shape)

### 7.2) Average attention over layers and heads

Compute the average attention by implementing the following function.

In [ ]:
# TODO: IMPLEMENT THIS CELL

def mean_attention_matrix(outputs) -> np.ndarray:
    """
    Average attention over all layers and heads.

    Returns:
        attention matrix of shape [seq_len, seq_len]
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY THIS CELL

attn_mean = mean_attention_matrix(outputs)

seq_len = attn_mean.shape[0]
assert attn_mean.shape == (seq_len, seq_len)
assert np.allclose(attn_mean.sum(axis=1), np.ones(seq_len), atol=1e-4)

print("Mean attention shape:", attn_mean.shape)
print("Mean attention row sums:", attn_mean.sum(axis=1))

### 7.3) Plot attention heatmap

Please plot the mean attention matrix as a heatmap. Include a title and a colorbar.


In [ ]:
# TODO: IMPLEMENT THIS CELL

# YOUR CODE HERE

### 7.4) Heavy hitter scores

For each key token position `j`, compute how much attention it receives in total

$$ 
\text{score}[j] = \sum_i A[i,j]
$$


In [ ]:
# TODO: IMPLEMENT THIS CELL

def heavy_hitter_scores(attn_matrix: np.ndarray) -> np.ndarray:
    """
    Compute accumulated attention received by each key token.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# DO NOT MODIFY THIS CELL

scores = heavy_hitter_scores(attn_mean)

assert scores.shape == (seq_len,)
assert np.all(scores >= 0)

tokens = attn_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for i, (tok, score) in enumerate(zip(tokens, scores)):
    print(i, repr(tok), float(score))

### 7.5) Plot heavy hitter scores

Plot heavy hitter scores by token position. 

In [ ]:
# YOUR CODE HERE

### 7.6) Implement KV Cache Policies
You will implement the following policies:
1. keep oldest
2. keep most recent
3. keep random
4. keep heavy hitters

These policies operate over token positions. They return a sorted list of token indices to keep.

In [ ]:
# TODO: IMPLEMENT THIS CELL

def keep_oldest(seq_len: int, budget: int) -> List[int]:
    """
    Keep the first `budget` token positions.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def keep_recent(seq_len: int, budget: int) -> List[int]:
    """
    Keep the most recent `budget` token positions.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def keep_random(seq_len: int, budget: int, seed: int = 42) -> List[int]:
    """
    Keep `budget` random token positions.
    Return sorted positions for deterministic comparison.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def keep_heavy_hitters(scores: np.ndarray, budget: int) -> List[int]:
    """
    Keep the `budget` token positions with highest heavy-hitter scores.
    Return sorted positions.
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# DO NOT MODIFY THIS CELL

toy_scores = np.array([10.0, 1.0, 5.0, 2.0, 20.0])
assert keep_oldest(5, 2) == [0, 1]
assert keep_recent(5, 2) == [3, 4]
assert keep_heavy_hitters(toy_scores, 2) == [0, 4]

rand_keep = keep_random(5, 3, seed=123)
assert len(rand_keep) == 3
assert len(set(rand_keep)) == 3
assert all(0 <= i < 5 for i in rand_keep)
assert rand_keep == sorted(rand_keep)



In [ ]:
budget = min(8, seq_len)

policies = {
    "oldest": keep_oldest(seq_len, budget),
    "recent": keep_recent(seq_len, budget),
    "random": keep_random(seq_len, budget, seed=SEED),
    "heavy hitters": keep_heavy_hitters(scores, budget),
}

for name, kept in policies.items():
    kept_tokens = [tokens[i] for i in kept]
    print(name)
    print("positions:", kept)
    print("tokens:", kept_tokens)
    print()